In [9]:
!pip install pymongo
import pandas as pd
from pymongo import MongoClient

In [10]:
# Connect to local MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["tmdb_movies"]
print("Connected to MongoDB!")

# Load the cleaned CSV
df = pd.read_csv("TMDB_cleaned_revenue.csv")
df = df.where(pd.notnull(df), None)
print(f"Loaded {len(df)} rows")

Connected to MongoDB!
Loaded 24404 rows


In [11]:
# Convert dataframe to list of dictionaries
movies = df.to_dict(orient='records')

# Insert into MongoDB collection
collection = db["movies"]
collection.drop()
collection.insert_many(movies)

print(f"Inserted {collection.count_documents({})} documents into MongoDB!")

Inserted 24404 documents into MongoDB!


In [12]:
# ── QUERY 1: Movies with highest audience engagement by genre ──
print("\n--- Query 1: Top 5 Most Voted Movies per Genre ---")
pipeline = [
    { "$match": { "vote_count": { "$gte": 1000 } } },
    { "$unwind": "$genres" },
    { "$sort": { "vote_count": -1 } },
    { "$group": {
        "_id": "$genres",
        "top_movie": { "$first": "$title" },
        "vote_count": { "$first": "$vote_count" },
        "revenue": { "$first": "$revenue" }
    }},
    { "$sort": { "vote_count": -1 } },
    { "$limit": 5 },

    { "$out": "top_movies_by_genre" }
]
results1 = list(collection.aggregate(pipeline))
for r in results1:
    print(r)

print("Saved Query 1 results to collection: top_movies_by_genre")

# ── QUERY 2: Average revenue by genre for movies with budget > 1M ──
print("\n--- Query 2: Average Revenue by Genre (budget > $1M) ---")
pipeline2 = [
    { "$match": { "budget": { "$gt": 1000000 } } },
    { "$unwind": "$genres" },
    { "$group": {
        "_id": "$genres",
        "avg_revenue": { "$avg": "$revenue" },
        "avg_budget": { "$avg": "$budget" },
        "movie_count": { "$sum": 1 }
    }},
    { "$sort": { "avg_revenue": -1 } },
    { "$limit": 10 },

    { "$out": "avg_revenue_by_genre" }
]
results2 = list(collection.aggregate(pipeline2))
for r in results2:
    print(r)

print("Saved Query 2 results to collection: avg_revenue_by_genre")


--- Query 1: Top 5 Most Voted Movies per Genre ---
Saved Query 1 results to collection: top_movies_by_genre

--- Query 2: Average Revenue by Genre (budget > $1M) ---
Saved Query 2 results to collection: avg_revenue_by_genre


In [13]:
# Question 1. Does a higher production budget lead to better outcomes?

print("\n--- Query 1: Budget vs Success ---")

pipeline1 = [

    {
        "$match": {
            "budget": { "$gt": 0 },
            "revenue": { "$gt": 0 }
        }
    },

    {
        "$bucket": {
            "groupBy": "$budget",
            "boundaries": [
                0,
                1000000,
                10000000,
                50000000,
                100000000,
                500000000
            ],
            "default": "500M+",

            "output": {
                "avg_revenue": { "$avg": "$revenue" },
                "avg_rating": { "$avg": "$vote_average" },
                "avg_votes": { "$avg": "$vote_count" },
                "movie_count": { "$sum": 1 }
            }
        }
    },

    {
        "$sort": { "_id": 1 }
    },

    # SAVE COLLECTION
    {
        "$out": "budget_vs_success"
    }
]

collection.aggregate(pipeline1)

print("Saved collection: budget_vs_success")


--- Query 1: Budget vs Success ---
Saved collection: budget_vs_success


In [17]:
#Question 2: Do movies with higher audience engagement tend to have higher revenue? Does it vary by genre?

print("\n--- Query 2: Engagement vs Revenue by Genre ---")

pipeline2 = [

    {
        "$match": {
            "vote_count": { "$gt": 100 },
            "revenue": { "$gt": 0 },
            "genres": { "$ne": None }
        }
    },

    # Convert comma-separated string into array
    {
        "$addFields": {
            "genre_array": {
                "$split": ["$genres", ", "]
            }
        }
    },

    # Split genres into separate rows
    {
        "$unwind": "$genre_array"
    },

    # Group by individual genre
    {
        "$group": {
            "_id": "$genre_array",

            "avg_vote_count": {
                "$avg": "$vote_count"
            },

            "avg_revenue": {
                "$avg": "$revenue"
            },

            "avg_rating": {
                "$avg": "$vote_average"
            },

            "movie_count": {
                "$sum": 1
            }
        }
    },

    {
        "$sort": {
            "avg_revenue": -1
        }
    },

    # SAVE COLLECTION
    {
        "$out": "engagement_vs_revenue_by_genre"
    }
]

collection.aggregate(pipeline2)

print("Saved collection: engagement_vs_revenue_by_genre")


--- Query 2: Engagement vs Revenue by Genre ---
Saved collection: engagement_vs_revenue_by_genre


In [15]:
#Question 3: What are the most popular genres in each country or region?

print("\n--- Query 3: Popular Genres by Country ---")

pipeline3 = [

    {
        "$match": {
            "production_countries": { "$ne": [] }
        }
    },

    { "$unwind": "$production_countries" },
    { "$unwind": "$genres" },

    {
        "$group": {
            "_id": {
                "country": "$production_countries",
                "genre": "$genres"
            },

            "avg_popularity": {
                "$avg": "$popularity"
            },

            "avg_vote_count": {
                "$avg": "$vote_count"
            },

            "movie_count": {
                "$sum": 1
            }
        }
    },

    {
        "$sort": {
            "avg_popularity": -1
        }
    },

    # SAVE COLLECTION
    {
        "$out": "popular_genres_by_country"
    }
]

collection.aggregate(pipeline3)

print("Saved collection: popular_genres_by_country")


--- Query 3: Popular Genres by Country ---
Saved collection: popular_genres_by_country


In [16]:
#Question 4: How has the film industry changed over time?

print("\n--- Query 4: Film Industry Trends Over Time ---")

pipeline4 = [

    {
        "$match": {
            "release_date": { "$ne": None }
        }
    },

    {
        "$addFields": {
            "year": {
                "$year": {
                    "$dateFromString": {
                        "dateString": "$release_date"
                    }
                }
            }
        }
    },

    {
        "$group": {
            "_id": "$year",

            "avg_budget": {
                "$avg": "$budget"
            },

            "avg_revenue": {
                "$avg": "$revenue"
            },

            "avg_rating": {
                "$avg": "$vote_average"
            },

            "avg_popularity": {
                "$avg": "$popularity"
            },

            "movie_count": {
                "$sum": 1
            }
        }
    },

    {
        "$sort": {
            "_id": 1
        }
    },

    # SAVE COLLECTION
    {
        "$out": "film_industry_trends_over_time"
    }
]

collection.aggregate(pipeline4)

print("Saved collection: film_industry_trends_over_time")


--- Query 4: Film Industry Trends Over Time ---


OperationFailure: Executor error during aggregate command on namespace: tmdb_movies.movies :: caused by :: $dateFromString requires that 'dateString' be a string, found: double with value nan, full error: {'ok': 0.0, 'errmsg': "Executor error during aggregate command on namespace: tmdb_movies.movies :: caused by :: $dateFromString requires that 'dateString' be a string, found: double with value nan", 'code': 241, 'codeName': 'ConversionFailure'}